# Binance Tick Data Library - Quickstart Guide

**Complete feature demonstration and practical examples**

This notebook provides comprehensive coverage of all major features in the Binance Tick Data library:

## Table of Contents

1. [Setup & Configuration](#1-setup--configuration)
2. [Data Repository Basics](#2-data-repository-basics)
3. [Querying Historical Data](#3-querying-historical-data)
4. [Dollar Volume Bar Sampling](#4-dollar-volume-bar-sampling)
5. [OHLCV Time-Based Sampling](#5-ohlcv-time-based-sampling)
6. [Market Microstructure Analysis](#6-market-microstructure-analysis)
7. [Volume Profile Analysis](#7-volume-profile-analysis)
8. [Order Flow Analysis](#8-order-flow-analysis)
9. [Liquidity Analysis](#9-liquidity-analysis)
10. [Real-Time Streaming](#10-real-time-streaming)
11. [Custom Analyzer Development](#11-custom-analyzer-development)
12. [Data Export](#12-data-export)
13. [Advanced Queries](#13-advanced-queries)
14. [Performance Optimization](#14-performance-optimization)
15. [Best Practices](#15-best-practices)

---

## 1. Setup & Configuration

First, let's import the necessary modules and set up the environment.

In [1]:
# Core imports
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import asyncio
from pathlib import Path

# Library imports - use repository.py which has all methods
from binance_tick_data.repository import BinanceDataRepository
from binance_tick_data.dollar_volume_sampling import (
    DollarVolumeSampler,
    create_dollar_volume_bars,
    calculate_optimal_threshold
)
from binance_tick_data.db_config import get_config
from binance_tick_data.errors import (
    BinanceDataError,
    NoDataFoundError,
    InvalidSymbolError
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.8f}'.format)

print("✓ All imports successful")

✓ All imports successful


In [2]:
import os; os.getcwd()

'/Users/mohamedali/trading_project/dlt-starter/notebooks'

In [3]:
os.chdir('..')

In [4]:
# Load configuration
config = get_config()

print(f"Database path: {config.database.db_path}")
print(f"Schema: {config.database.schema}")
print(f"Catalog: {config.database.catalog}")
print(f"\nAvailable tables:")
print(f"  - Aggregated trades: {config.get_table_path('agg_trades')}")
print(f"  - Trades: {config.get_table_path('trades')}")
print(f"  - Order book: {config.get_table_path('order_book_snapshots')}")

Database path: binance_pipeline.duckdb
Schema: binance_data
Catalog: 

Available tables:
  - Aggregated trades: binance_data.agg_trades
  - Trades: binance_data.trades
  - Order book: binance_data.order_book_snapshots


## 2. Data Repository Basics

The `BinanceDataRepository` is the main interface for querying data.

In [5]:
# Initialize repository (read-only mode)
# The old repository.py accepts db_path and read_only parameters directly
config = get_config()
repo = BinanceDataRepository(
    db_path=config.database.db_path,
    dataset_name=config.database.schema_name,
    read_only=True
)

print("Repository initialized")
print(f"Database: {repo.db_path}")
print(f"Schema: {repo.dataset_name}")
print(f"Mode: {'Read-only' if repo.read_only else 'Read-write'}")

Repository initialized
Database: binance_pipeline.duckdb
Schema: binance_data
Mode: Read-only


In [6]:
# Check available symbols
# This query depends on your database structure
symbols = ['BTCUSDT', 'ETHUSDT']  # Common symbols

print(f"Working with symbols: {symbols}")

Working with symbols: ['BTCUSDT', 'ETHUSDT']


## 3. Querying Historical Data

### 3.1 Get Recent Aggregated Trades

In [7]:
# Get last 7 days of BTCUSDT trades
symbol = 'BTCUSDT'
days_lookback = 7

try:
    df_trades = repo.get_agg_trades_by_date_range(
        symbol=symbol,
        days=days_lookback,
        as_dataframe=True
    )
    
    print(f"✓ Retrieved {len(df_trades):,} trades for {symbol}")
    print(f"\nDate range: {df_trades['timestamp'].min()} to {df_trades['timestamp'].max()}")
    print(f"\nData shape: {df_trades.shape}")
    print(f"\nFirst few rows:")
    display(df_trades.head())
    
except NoDataFoundError as e:
    print(f"⚠️  No data found: {e}")
except BinanceDataError as e:
    print(f"❌ Error: {e}")

✓ Retrieved 0 trades for BTCUSDT

Date range: nan to nan

Data shape: (0, 9)

First few rows:


,agg_trade_id,symbol,price,quantity,first_trade_id,last_trade_id,timestamp,is_buyer_maker,is_best_match


### 3.2 Get Trades with Specific Time Range

In [8]:
# Define time range - use same range as loaded data
# Get date range from the loaded data
if len(df_trades) > 0:
    start_time = df_trades['timestamp'].min()
    end_time = df_trades['timestamp'].max()
    print(f"Using data time range: {start_time} to {end_time}")
else:
    # Fallback if no data
    end_time = datetime.now()
    start_time = end_time - timedelta(days=days_lookback)
    print(f"No data loaded, using fallback range: {start_time} to {end_time}")

# Query trades within the range (this demonstrates time-based querying)
df_trades_range = repo.get_agg_trades(
    symbol=symbol,
    start_time=start_time,
    end_time=end_time,
    limit=100000,
    as_dataframe=True
)

print(f"\nRetrieved {len(df_trades_range):,} trades from time range")
print(f"\nColumns: {list(df_trades_range.columns)}")
print(f"\nSample data:")
display(df_trades_range.head(10))

No data loaded, using fallback range: 2025-11-11 13:26:06.552660 to 2025-11-18 13:26:06.552660

Retrieved 0 trades from time range

Columns: ['agg_trade_id', 'symbol', 'price', 'quantity', 'first_trade_id', 'last_trade_id', 'timestamp', 'is_buyer_maker', 'is_best_match']

Sample data:


,agg_trade_id,symbol,price,quantity,first_trade_id,last_trade_id,timestamp,is_buyer_maker,is_best_match


### 3.3 Get Symbol Statistics

In [9]:
# Get comprehensive statistics
# Use same time range as the data we loaded (last 7 days)
stats = repo.get_symbol_stats(
    symbol=symbol,
    # Using date range from loaded data, not real-time
    # start_time and end_time can be omitted to get all data
)

# Display with rich formatting (uses _repr_html_ in Jupyter)
stats

SymbolStats(symbol='BTCUSDT', trades=110,000, price_range=$107014.58-$114551.76)

## 4. Dollar Volume Bar Sampling

Dollar volume bars are **information-driven** sampling methods that create bars based on transaction value instead of time.

### 4.1 Fixed Threshold Dollar Bars

### 3.4 Prepare Data for Dollar Volume Sampling

DuckDB returns price and quantity as strings (VARCHAR). We need to convert them to float for numerical operations.

In [10]:
# Convert string types to numeric (DuckDB returns VARCHAR)
df_trades['price'] = df_trades['price'].astype(float)
df_trades['quantity'] = df_trades['quantity'].astype(float)

print("✅ Converted price and quantity to float")
print(f"Price dtype: {df_trades["price"].dtype}")
print(f"Quantity dtype: {df_trades["quantity"].dtype}")

✅ Converted price and quantity to float
Price dtype: float64
Quantity dtype: float64


In [11]:
# Create dollar volume sampler with $1M threshold
threshold = 1_000_000  # $1 million USD per bar

sampler = DollarVolumeSampler(
    threshold=threshold,
    ticks_per_bar=100,
    adaptive=False
)

# Create bars from trade data
df_dollar_bars = sampler.create_bars(
    df_trades,
    price_col='price',
    volume_col='quantity',
    timestamp_col='timestamp'
)

print(f"✓ Created {len(df_dollar_bars):,} dollar volume bars")
print(f"\nBar columns: {list(df_dollar_bars.columns)}")
print(f"\nFirst 5 bars:")
display(df_dollar_bars.head())

# Verify dollar volume
print(f"\nDollar volume per bar - Statistics:")
print(df_dollar_bars['dollar_volume'].describe())

InsufficientDataError: ❌ Insufficient data for operation: DataFrame is empty

📋 Details:
   • required_records: 1
   • available_records: 0
   • shortage: 1

💡 Suggestions:
   → Need at least 1 records, but only 0 available
   → Fetch more historical data
   → Adjust the operation parameters

### 4.2 Adaptive Threshold Dollar Bars

In [12]:
# Adaptive sampler adjusts threshold based on recent bars
adaptive_sampler = DollarVolumeSampler(
    threshold=1_000_000,
    ticks_per_bar=100,
    adaptive=True,
    lookback_bars=20  # Use last 20 bars for threshold adjustment
)

df_adaptive_bars = adaptive_sampler.create_bars(
    df_trades,
    price_col='price',
    volume_col='quantity',
    timestamp_col='timestamp'
)

print(f"✓ Created {len(df_adaptive_bars):,} adaptive dollar volume bars")
print(f"\nAdaptive vs Fixed comparison:")
print(f"  Fixed bars: {len(df_dollar_bars):,}")
print(f"  Adaptive bars: {len(df_adaptive_bars):,}")

✓ Created 10 adaptive dollar volume bars

Adaptive vs Fixed comparison:
  Fixed bars: 12
  Adaptive bars: 10


### 4.3 Optimal Threshold Calculation

In [13]:
# Calculate optimal threshold from data
optimal_threshold = calculate_optimal_threshold(
    df_trades,
    price_col='price',
    volume_col='quantity',
    target_bars_per_day=50  # Aim for 50 bars per day
)

print(f"Optimal threshold: ${optimal_threshold:,.2f}")

# Use optimal threshold
df_optimal_bars = create_dollar_volume_bars(
    df_trades,
    threshold=optimal_threshold,
    price_col='price',
    volume_col='quantity',
    timestamp_col='timestamp'
)

print(f"✓ Created {len(df_optimal_bars):,} bars with optimal threshold")

TypeError: calculate_optimal_threshold() got an unexpected keyword argument 'target_bars_per_day'

### 4.4 Visualize Dollar Bars

In [ ]:
# Plot dollar bars as candlesticks
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Price chart
ax1 = axes[0]
ax1.plot(df_dollar_bars.index, df_dollar_bars['close'], label='Close', linewidth=1)
ax1.fill_between(df_dollar_bars.index, df_dollar_bars['low'], df_dollar_bars['high'], alpha=0.3, label='High-Low Range')
ax1.set_ylabel('Price (USD)')
ax1.set_title(f'{symbol} - Dollar Volume Bars (${threshold:,.0f} per bar)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Volume
ax2 = axes[1]
ax2.bar(df_dollar_bars.index, df_dollar_bars['volume'], alpha=0.7, color='steelblue')
ax2.set_ylabel('Volume')
ax2.set_title('Volume per Bar')
ax2.grid(True, alpha=0.3)

# Tick count
ax3 = axes[2]
ax3.bar(df_dollar_bars.index, df_dollar_bars['tick_count'], alpha=0.7, color='coral')
ax3.set_ylabel('Tick Count')
ax3.set_xlabel('Bar Number')
ax3.set_title('Ticks per Bar')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nBar statistics:")
print(df_dollar_bars[['volume', 'dollar_volume', 'tick_count', 'vwap']].describe())

### 4.5 Statistical Properties of Dollar Bars

In [ ]:
# Calculate returns
df_dollar_bars['returns'] = df_dollar_bars['close'].pct_change()
df_dollar_bars['log_returns'] = np.log(df_dollar_bars['close'] / df_dollar_bars['close'].shift(1))

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Returns histogram
ax1 = axes[0]
df_dollar_bars['returns'].dropna().hist(bins=50, ax=ax1, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Returns')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Dollar Bar Returns')
ax1.axvline(0, color='red', linestyle='--', linewidth=1)
ax1.grid(True, alpha=0.3)

# Q-Q plot
from scipy import stats
ax2 = axes[1]
stats.probplot(df_dollar_bars['returns'].dropna(), dist="norm", plot=ax2)
ax2.set_title('Q-Q Plot (Normal Distribution)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical tests
from scipy.stats import normaltest, jarque_bera

returns_clean = df_dollar_bars['returns'].dropna()
stat_normal, p_normal = normaltest(returns_clean)
stat_jb, p_jb = jarque_bera(returns_clean)

print("\nNormality Tests:")
print(f"  Normal test: stat={stat_normal:.4f}, p-value={p_normal:.4f}")
print(f"  Jarque-Bera: stat={stat_jb:.4f}, p-value={p_jb:.4f}")
print(f"\nReturns statistics:")
print(f"  Mean: {returns_clean.mean():.6f}")
print(f"  Std: {returns_clean.std():.6f}")
print(f"  Skewness: {returns_clean.skew():.4f}")
print(f"  Kurtosis: {returns_clean.kurtosis():.4f}")

## 5. OHLCV Time-Based Sampling

Traditional time-based candles using the repository's built-in aggregation.

In [ ]:
# Get 1-hour OHLCV data
df_1h = repo.get_ohlcv(
    symbol=symbol,
    interval='1h',
    start_time=start_time,
    end_time=end_time,
    return_dataframe=True
)

print(f"✓ Retrieved {len(df_1h):,} hourly candles")
display(df_1h.head())

# Plot OHLCV
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Price
ax1 = axes[0]
ax1.plot(df_1h['timestamp'], df_1h['close'], label='Close', linewidth=1.5)
ax1.fill_between(df_1h['timestamp'], df_1h['low'], df_1h['high'], alpha=0.2)
ax1.set_ylabel('Price (USD)')
ax1.set_title(f'{symbol} - 1H Candles')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Volume
ax2 = axes[1]
ax2.bar(df_1h['timestamp'], df_1h['volume'], width=0.03, alpha=0.7)
ax2.set_ylabel('Volume')
ax2.set_xlabel('Time')
ax2.set_title('Volume')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare multiple intervals
intervals = ['5m', '15m', '1h', '4h']
interval_data = {}

for interval in intervals:
    df_interval = repo.get_ohlcv(
        symbol=symbol,
        interval=interval,
        start_time=start_time,
        end_time=end_time,
        return_dataframe=True
    )
    interval_data[interval] = df_interval
    print(f"{interval:5s}: {len(df_interval):4d} candles")

# Plot comparison
fig, axes = plt.subplots(len(intervals), 1, figsize=(14, 12))

for idx, (interval, df) in enumerate(interval_data.items()):
    ax = axes[idx]
    ax.plot(df['timestamp'], df['close'], linewidth=1)
    ax.set_ylabel('Price')
    ax.set_title(f'{interval} Interval')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Market Microstructure Analysis

Analyze trade-level microstructure patterns.

In [ ]:
# Analyze buy vs sell pressure
df_trades['side'] = df_trades['is_buyer_maker'].apply(lambda x: 'sell' if x else 'buy')
df_trades['dollar_volume'] = df_trades['price'] * df_trades['quantity']

# Aggregate by side
side_summary = df_trades.groupby('side').agg({
    'quantity': ['count', 'sum', 'mean'],
    'dollar_volume': ['sum', 'mean'],
    'price': ['mean', 'std']
})

print("Buy vs Sell Analysis:")
print("="*60)
display(side_summary)

# Calculate imbalance
buy_volume = df_trades[df_trades['side'] == 'buy']['dollar_volume'].sum()
sell_volume = df_trades[df_trades['side'] == 'sell']['dollar_volume'].sum()
imbalance = (buy_volume - sell_volume) / (buy_volume + sell_volume)

print(f"\nOrder Imbalance: {imbalance:.4f}")
print(f"  Buy volume: ${buy_volume:,.2f}")
print(f"  Sell volume: ${sell_volume:,.2f}")
print(f"  Interpretation: {'Buying pressure' if imbalance > 0 else 'Selling pressure'}")

In [ ]:
# Visualize order flow over time
df_trades.set_index('timestamp', inplace=True)

# Resample to 5-minute windows
buy_flow = df_trades[df_trades['side'] == 'buy'].resample('5min')['dollar_volume'].sum()
sell_flow = df_trades[df_trades['side'] == 'sell'].resample('5min')['dollar_volume'].sum()
net_flow = buy_flow - sell_flow

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Stacked area chart
ax1 = axes[0]
ax1.fill_between(buy_flow.index, 0, buy_flow, alpha=0.6, color='green', label='Buy Volume')
ax1.fill_between(sell_flow.index, 0, -sell_flow, alpha=0.6, color='red', label='Sell Volume')
ax1.set_ylabel('Dollar Volume')
ax1.set_title('Order Flow (5-min intervals)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axhline(0, color='black', linewidth=0.8)

# Net flow
ax2 = axes[1]
colors = ['green' if x > 0 else 'red' for x in net_flow]
ax2.bar(net_flow.index, net_flow, width=0.003, alpha=0.7, color=colors)
ax2.set_ylabel('Net Flow')
ax2.set_xlabel('Time')
ax2.set_title('Net Order Flow (Buy - Sell)')
ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

df_trades.reset_index(inplace=True)

## 7. Volume Profile Analysis

Analyze volume distribution across price levels.

In [ ]:
# Get volume profile from repository
df_volume_profile = repo.get_volume_profile(
    symbol=symbol,
    price_bins=50,
    start_time=start_time,
    end_time=end_time,
    return_dataframe=True
)

print(f"✓ Volume profile with {len(df_volume_profile)} price bins")
display(df_volume_profile.head())

# Calculate POC (Point of Control)
poc_idx = df_volume_profile['total_volume'].idxmax()
poc_price = df_volume_profile.loc[poc_idx, 'price_level']
poc_volume = df_volume_profile.loc[poc_idx, 'total_volume']

print(f"\nPoint of Control (POC):")
print(f"  Price: ${poc_price:,.2f}")
print(f"  Volume: {poc_volume:,.2f}")

In [ ]:
# Visualize volume profile
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Horizontal volume profile
ax1 = axes[0]
ax1.barh(df_volume_profile['price_level'], df_volume_profile['total_volume'], alpha=0.7)
ax1.axhline(poc_price, color='red', linestyle='--', linewidth=2, label='POC')
ax1.set_xlabel('Volume')
ax1.set_ylabel('Price (USD)')
ax1.set_title('Volume Profile')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Buy vs sell volume
ax2 = axes[1]
ax2.barh(df_volume_profile['price_level'], df_volume_profile['buy_volume'], 
         alpha=0.7, color='green', label='Buy Volume')
ax2.barh(df_volume_profile['price_level'], -df_volume_profile['sell_volume'], 
         alpha=0.7, color='red', label='Sell Volume')
ax2.axhline(poc_price, color='black', linestyle='--', linewidth=2, label='POC')
ax2.set_xlabel('Volume (Buy + / Sell -)')
ax2.set_ylabel('Price (USD)')
ax2.set_title('Buy vs Sell Volume Profile')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Order Flow Analysis

Deep dive into order flow dynamics using the OrderFlowAnalyzer pattern.

In [ ]:
# Manual order flow analysis (without async analyzer)
from collections import deque

def analyze_order_flow(trades_df, window_minutes=5):
    """Analyze order flow in time windows"""
    
    trades_df = trades_df.copy()
    trades_df.set_index('timestamp', inplace=True)
    
    # Resample to windows
    window_str = f'{window_minutes}min'
    
    # Calculate metrics per window
    buy_trades = trades_df[trades_df['side'] == 'buy']
    sell_trades = trades_df[trades_df['side'] == 'sell']
    
    metrics = pd.DataFrame({
        'buy_count': buy_trades.resample(window_str).size(),
        'sell_count': sell_trades.resample(window_str).size(),
        'buy_volume': buy_trades.resample(window_str)['quantity'].sum(),
        'sell_volume': sell_trades.resample(window_str)['quantity'].sum(),
        'buy_dollar_volume': buy_trades.resample(window_str)['dollar_volume'].sum(),
        'sell_dollar_volume': sell_trades.resample(window_str)['dollar_volume'].sum(),
        'vwap': trades_df.resample(window_str).apply(
            lambda x: (x['price'] * x['quantity']).sum() / x['quantity'].sum() if x['quantity'].sum() > 0 else np.nan
        )
    })
    
    # Calculate imbalances
    metrics['order_imbalance'] = (
        (metrics['buy_dollar_volume'] - metrics['sell_dollar_volume']) / 
        (metrics['buy_dollar_volume'] + metrics['sell_dollar_volume'])
    )
    
    metrics['trade_imbalance'] = (
        (metrics['buy_count'] - metrics['sell_count']) / 
        (metrics['buy_count'] + metrics['sell_count'])
    )
    
    metrics.reset_index(inplace=True)
    return metrics

# Analyze
flow_metrics = analyze_order_flow(df_trades, window_minutes=5)

print(f"✓ Order flow metrics calculated for {len(flow_metrics)} windows")
display(flow_metrics.head(10))

In [ ]:
# Visualize order flow metrics
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Order imbalance
ax1 = axes[0]
colors = ['green' if x > 0 else 'red' for x in flow_metrics['order_imbalance']]
ax1.bar(flow_metrics['timestamp'], flow_metrics['order_imbalance'], 
        width=0.003, alpha=0.7, color=colors)
ax1.set_ylabel('Imbalance')
ax1.set_title('Order Imbalance (Dollar Volume)')
ax1.axhline(0, color='black', linewidth=0.8)
ax1.grid(True, alpha=0.3)

# Trade count imbalance
ax2 = axes[1]
colors = ['green' if x > 0 else 'red' for x in flow_metrics['trade_imbalance']]
ax2.bar(flow_metrics['timestamp'], flow_metrics['trade_imbalance'], 
        width=0.003, alpha=0.7, color=colors)
ax2.set_ylabel('Imbalance')
ax2.set_title('Trade Count Imbalance')
ax2.axhline(0, color='black', linewidth=0.8)
ax2.grid(True, alpha=0.3)

# VWAP
ax3 = axes[2]
ax3.plot(flow_metrics['timestamp'], flow_metrics['vwap'], linewidth=1.5)
ax3.set_ylabel('Price (USD)')
ax3.set_xlabel('Time')
ax3.set_title('5-min VWAP')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Liquidity Analysis

Estimate liquidity from trade data.

In [ ]:
# Calculate effective spread and liquidity metrics
def calculate_liquidity_metrics(trades_df, window_minutes=5):
    """Calculate liquidity metrics from trades"""
    
    trades_df = trades_df.copy()
    trades_df.set_index('timestamp', inplace=True)
    window_str = f'{window_minutes}min'
    
    metrics = pd.DataFrame({
        'trade_count': trades_df.resample(window_str).size(),
        'avg_trade_size': trades_df.resample(window_str)['quantity'].mean(),
        'price_std': trades_df.resample(window_str)['price'].std(),
        'price_range': trades_df.resample(window_str)['price'].apply(lambda x: x.max() - x.min()),
        'mid_price': trades_df.resample(window_str)['price'].mean(),
    })
    
    # Spread estimate (price range as proxy)
    metrics['spread_estimate'] = metrics['price_range'] / metrics['mid_price'] * 10000  # in basis points
    
    # Liquidity score (high trade count + low volatility = high liquidity)
    metrics['liquidity_score'] = metrics['trade_count'] / (metrics['price_std'].fillna(1) + 1e-6)
    
    metrics.reset_index(inplace=True)
    return metrics

liquidity_metrics = calculate_liquidity_metrics(df_trades, window_minutes=5)

print(f"✓ Liquidity metrics calculated")
display(liquidity_metrics.head(10))

print("\nSummary statistics:")
print(liquidity_metrics[['trade_count', 'avg_trade_size', 'spread_estimate', 'liquidity_score']].describe())

In [ ]:
# Visualize liquidity
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Spread estimate
ax1 = axes[0]
ax1.plot(liquidity_metrics['timestamp'], liquidity_metrics['spread_estimate'], linewidth=1)
ax1.set_ylabel('Basis Points')
ax1.set_title('Estimated Spread (basis points)')
ax1.grid(True, alpha=0.3)

# Trade activity
ax2 = axes[1]
ax2.bar(liquidity_metrics['timestamp'], liquidity_metrics['trade_count'], 
        width=0.003, alpha=0.7, color='steelblue')
ax2.set_ylabel('Count')
ax2.set_title('Trade Count per 5-min Window')
ax2.grid(True, alpha=0.3)

# Liquidity score
ax3 = axes[2]
ax3.plot(liquidity_metrics['timestamp'], liquidity_metrics['liquidity_score'], linewidth=1, color='green')
ax3.set_ylabel('Score')
ax3.set_xlabel('Time')
ax3.set_title('Liquidity Score (Trade Frequency / Volatility)')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Real-Time Streaming

Demonstrate real-time data consumption using the RealtimeConsumer.

**Note:** This requires an active internet connection and runs asynchronously.

In [ ]:
# Import real-time components
from binance_tick_data.consumers.realtime_consumer import RealtimeConsumer, ConsumerConfig
from binance_tick_data.analyzers.order_flow_analyzer import OrderFlowAnalyzer
from binance_tick_data.analyzers.liquidity_analyzer import LiquidityAnalyzer
from binance_tick_data.analyzers.volume_profile_analyzer import VolumeProfileAnalyzer

print("✓ Real-time components imported")

In [ ]:
# Configure consumer
realtime_config = ConsumerConfig(
    symbols=['BTCUSDT', 'ETHUSDT'],
    buffer_size=10000,
    update_interval=0.1,
    stream_orderbook=False,  # Set True to stream order book
    batch_size=100
)

# Create consumer
consumer = RealtimeConsumer(realtime_config)

# Register analyzers
consumer.register_analyzer(OrderFlowAnalyzer(window_size=60))  # 60-second windows
consumer.register_analyzer(LiquidityAnalyzer(window_size=60))
consumer.register_analyzer(VolumeProfileAnalyzer(window_size=60, price_bins=20))

print("✓ Consumer configured with 3 analyzers")
print(f"  Symbols: {realtime_config.symbols}")
print(f"  Buffer size: {realtime_config.buffer_size:,}")

In [ ]:
# Run consumer for 30 seconds
async def stream_demo(duration_seconds=30):
    """Stream data and display statistics"""
    
    print(f"Starting real-time stream for {duration_seconds} seconds...\n")
    
    await consumer.start()
    
    # Monitor for duration
    await asyncio.sleep(duration_seconds)
    
    # Get statistics
    stats = consumer.get_statistics()
    
    print("\nStream Statistics:")
    print("="*60)
    for symbol, symbol_stats in stats.items():
        print(f"\n{symbol}:")
        for key, value in symbol_stats.items():
            print(f"  {key:30s}: {value}")
    
    # Get recent trades
    recent_btc = consumer.get_recent_trades('BTCUSDT', n=10)
    print(f"\nRecent BTCUSDT trades: {len(recent_btc)}")
    if recent_btc:
        for trade in recent_btc[:5]:
            print(f"  {trade.time}: ${trade.price:,.2f} x {trade.qty:.4f}")
    
    await consumer.stop()
    print("\n✓ Stream stopped")

# Run
# Uncomment to run the stream
# await stream_demo(duration_seconds=30)

## 11. Custom Analyzer Development

Create a custom analyzer for specific needs.

In [ ]:
from binance_tick_data.analyzers.base import BaseAnalyzer
from binance_tick_data.sources.schemas import Trade, AggTrade
from typing import Dict, Optional
from datetime import datetime

class CustomMomentumAnalyzer(BaseAnalyzer):
    """
    Custom analyzer to track price momentum and acceleration.
    """
    
    def __init__(self, window_size: int = 60):
        super().__init__(name="custom_momentum", window_size=window_size)
        self.prices = []
        self.volumes = []
        self.timestamps = []
    
    async def on_trade(self, trade: Trade) -> Optional[Dict]:
        """Process each trade"""
        self.prices.append(trade.price)
        self.volumes.append(trade.qty)
        self.timestamps.append(trade.time)
        return None
    
    async def on_window_close(self, window_end: datetime) -> Dict:
        """Calculate momentum metrics at window close"""
        if len(self.prices) < 2:
            return {}
        
        prices = np.array(self.prices)
        volumes = np.array(self.volumes)
        
        # Calculate metrics
        price_change = prices[-1] - prices[0]
        price_change_pct = (price_change / prices[0]) * 100
        
        # Momentum (rate of change)
        if len(prices) > 10:
            recent_change = prices[-1] - prices[-10]
            momentum = recent_change / prices[-10] * 100
        else:
            momentum = price_change_pct
        
        # Acceleration (change in momentum)
        if len(prices) > 20:
            prev_momentum = (prices[-10] - prices[-20]) / prices[-20] * 100
            acceleration = momentum - prev_momentum
        else:
            acceleration = 0
        
        # Volume-weighted momentum
        vw_price = np.average(prices, weights=volumes)
        
        metrics = {
            'price_change': float(price_change),
            'price_change_pct': float(price_change_pct),
            'momentum': float(momentum),
            'acceleration': float(acceleration),
            'vwap': float(vw_price),
            'volatility': float(np.std(prices)),
            'trade_count': len(self.prices)
        }
        
        # Reset for next window
        self.prices = []
        self.volumes = []
        self.timestamps = []
        
        return metrics
    
    def get_current_metrics(self) -> Dict:
        """Get current metrics without closing window"""
        if len(self.prices) < 2:
            return {}
        
        prices = np.array(self.prices)
        return {
            'current_price': float(prices[-1]),
            'window_change': float(prices[-1] - prices[0]),
            'trade_count': len(self.prices)
        }

print("✓ Custom analyzer defined")
print("\nFeatures:")
print("  - Price change tracking")
print("  - Momentum calculation")
print("  - Acceleration (momentum derivative)")
print("  - Volume-weighted average price")
print("  - Volatility measurement")

In [ ]:
# Test custom analyzer on historical data (simulation)
def simulate_analyzer(analyzer, trades_df, window_size=60):
    """Simulate analyzer on historical data"""
    
    trades_df = trades_df.copy().sort_values('timestamp')
    
    results = []
    window_trades = []
    window_start = trades_df['timestamp'].iloc[0]
    
    for _, row in trades_df.iterrows():
        # Create Trade object
        trade = Trade(
            id=int(row.get('agg_trade_id', 0)),
            price=float(row['price']),
            qty=float(row['quantity']),
            quote_qty=float(row['price']) * float(row['quantity']),
            time=row['timestamp'],
            is_buyer_maker=bool(row.get('is_buyer_maker', False)),
            is_best_match=True,
            symbol=symbol
        )
        
        # Check if window should close
        if (row['timestamp'] - window_start).total_seconds() >= window_size:
            # Close window
            metrics = asyncio.run(analyzer.on_window_close(row['timestamp']))
            if metrics:
                metrics['timestamp'] = window_start
                results.append(metrics)
            
            window_start = row['timestamp']
        
        # Process trade
        asyncio.run(analyzer.on_trade(trade))
    
    return pd.DataFrame(results)

# Run simulation
momentum_analyzer = CustomMomentumAnalyzer(window_size=60)
momentum_results = simulate_analyzer(momentum_analyzer, df_trades, window_size=60)

print(f"✓ Analyzed {len(momentum_results)} windows")
display(momentum_results.head(10))

In [ ]:
# Visualize custom analyzer results
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Momentum
ax1 = axes[0]
colors = ['green' if x > 0 else 'red' for x in momentum_results['momentum']]
ax1.bar(momentum_results['timestamp'], momentum_results['momentum'], 
        width=0.0005, alpha=0.7, color=colors)
ax1.set_ylabel('Momentum (%)')
ax1.set_title('Price Momentum (60s windows)')
ax1.axhline(0, color='black', linewidth=0.8)
ax1.grid(True, alpha=0.3)

# Acceleration
ax2 = axes[1]
colors = ['green' if x > 0 else 'red' for x in momentum_results['acceleration']]
ax2.bar(momentum_results['timestamp'], momentum_results['acceleration'], 
        width=0.0005, alpha=0.7, color=colors)
ax2.set_ylabel('Acceleration (%)')
ax2.set_title('Momentum Acceleration')
ax2.axhline(0, color='black', linewidth=0.8)
ax2.grid(True, alpha=0.3)

# Volatility
ax3 = axes[2]
ax3.plot(momentum_results['timestamp'], momentum_results['volatility'], linewidth=1)
ax3.set_ylabel('Volatility')
ax3.set_xlabel('Time')
ax3.set_title('Price Volatility (Standard Deviation)')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Data Export

Export data to various formats for external analysis.

In [ ]:
# Create output directory
output_dir = Path('./output')
output_dir.mkdir(exist_ok=True)

print(f"Output directory: {output_dir.absolute()}")

In [ ]:
# Export to Parquet (efficient columnar format)
parquet_path = output_dir / f"{symbol}_trades.parquet"

repo.export_to_parquet(
    symbol=symbol,
    output_path=str(parquet_path),
    table='agg_trades',
    start_time=start_time,
    end_time=end_time
)

print(f"✓ Exported to: {parquet_path}")
print(f"  File size: {parquet_path.stat().st_size / 1024 / 1024:.2f} MB")

In [ ]:
# Export to CSV
csv_path = output_dir / f"{symbol}_trades.csv"

repo.export_to_csv(
    symbol=symbol,
    output_path=str(csv_path),
    table='agg_trades',
    start_time=start_time,
    end_time=end_time
)

print(f"✓ Exported to: {csv_path}")
print(f"  File size: {csv_path.stat().st_size / 1024 / 1024:.2f} MB")

In [ ]:
# Export dollar bars
dollar_bars_path = output_dir / f"{symbol}_dollar_bars.parquet"
df_dollar_bars.to_parquet(dollar_bars_path, compression='zstd')

print(f"✓ Exported dollar bars to: {dollar_bars_path}")
print(f"  File size: {dollar_bars_path.stat().st_size / 1024:.2f} KB")

## 13. Advanced Queries

Demonstrate advanced query patterns.

In [ ]:
# Get order book snapshot
snapshot_time = df_trades['timestamp'].iloc[len(df_trades)//2]  # Middle of time range

try:
    snapshot = repo.get_order_book_snapshot(
        symbol=symbol,
        timestamp=snapshot_time
    )
    
    print(f"✓ Order book snapshot at {snapshot_time}")
    print(f"\nBids:")
    for i, (price, qty) in enumerate(snapshot.get('bids', [])[:5]):
        print(f"  {i+1}. ${price:,.2f} x {qty:.4f}")
    
    print(f"\nAsks:")
    for i, (price, qty) in enumerate(snapshot.get('asks', [])[:5]):
        print(f"  {i+1}. ${price:,.2f} x {qty:.4f}")
        
except Exception as e:
    print(f"⚠️  Order book data not available: {e}")

In [ ]:
# Multi-symbol comparison
symbols_compare = ['BTCUSDT', 'ETHUSDT']
multi_stats = {}

for sym in symbols_compare:
    try:
        stats = repo.get_symbol_stats(
            symbol=sym,
            start_time=start_time,
            end_time=end_time
        )
        multi_stats[sym] = stats.to_dict()  # Convert to dict for DataFrame
    except Exception as e:
        print(f"⚠️  Could not get stats for {sym}: {e}")

# Compare
if len(multi_stats) > 1:
    comparison = pd.DataFrame(multi_stats).T
    print("\nMulti-Symbol Comparison:")
    print("="*80)
    display(comparison)

## 14. Performance Optimization

Tips and techniques for optimal performance.

In [ ]:
import time

# Benchmark: DataFrame vs tuples
print("Performance Comparison: DataFrame vs Tuples\n")

# Test with DataFrame
start = time.time()
df_result = repo.get_agg_trades(
    symbol=symbol,
    start_time=start_time,
    end_time=end_time,
    limit=50000,
    return_dataframe=True
)
df_time = time.time() - start

print(f"DataFrame query: {df_time:.4f}s ({len(df_result):,} rows)")

# Test with tuples
start = time.time()
tuple_result = repo.get_agg_trades(
    symbol=symbol,
    start_time=start_time,
    end_time=end_time,
    limit=50000,
    return_dataframe=False
)
tuple_time = time.time() - start

print(f"Tuple query: {tuple_time:.4f}s ({len(tuple_result):,} rows)")
print(f"\nSpeedup: {df_time/tuple_time:.2f}x faster with tuples")

print("\n💡 Tip: Use tuples for large queries if you'll convert to DataFrame anyway")

In [ ]:
# Benchmark: Limit parameter
print("Impact of LIMIT on query performance\n")

limits = [1000, 10000, 50000, 100000]
times = []

for limit in limits:
    start = time.time()
    result = repo.get_agg_trades(
        symbol=symbol,
        start_time=start_time,
        end_time=end_time,
        limit=limit,
        return_dataframe=False
    )
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"Limit {limit:6,}: {elapsed:.4f}s ({len(result):,} rows)")

# Plot
plt.figure(figsize=(10, 5))
plt.plot(limits, times, marker='o', linewidth=2, markersize=8)
plt.xlabel('Limit')
plt.ylabel('Query Time (seconds)')
plt.title('Query Performance vs Limit')
plt.grid(True, alpha=0.3)
plt.show()

print("\n💡 Tip: Use appropriate limits to balance performance and data completeness")

## 15. Best Practices

### Error Handling

In [ ]:
# Proper error handling
from binance_tick_data.errors import (
    NoDataFoundError,
    InvalidSymbolError,
    DatabaseError
)

def safe_query_example(symbol, days=7):
    """Example of proper error handling"""
    
    try:
        df = repo.get_agg_trades_by_date_range(
            symbol=symbol,
            days=days,
            return_dataframe=True
        )
        print(f"✓ Successfully retrieved {len(df):,} trades")
        return df
        
    except NoDataFoundError as e:
        print(f"⚠️  No data found: {e}")
        print("   Try a different time range or symbol")
        return None
        
    except InvalidSymbolError as e:
        print(f"❌ Invalid symbol: {e}")
        print("   Check symbol format (e.g., BTCUSDT)")
        return None
        
    except DatabaseError as e:
        print(f"❌ Database error: {e}")
        print("   Check database connection and permissions")
        return None
        
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        return None

# Test
result = safe_query_example('BTCUSDT', days=7)
if result is not None:
    print(f"Data shape: {result.shape}")

### Context Manager Usage

In [ ]:
# Always use context manager for automatic cleanup
def context_manager_example():
    """Proper usage of repository as context manager"""
    
    config = get_config()
    with BinanceDataRepository(
        db_path=config.database.db_path,
        dataset_name=config.database.schema_name,
        read_only=True
    ) as repo:
        # Repository automatically connects
        stats = repo.get_symbol_stats('BTCUSDT')
        
        # Do work...
        
        # Repository automatically closes
        return stats
    
print("✓ Context manager ensures proper cleanup")
print("  - Automatic connection management")
print("  - Guaranteed cleanup even on errors")
print("  - Prevents resource leaks")

### Memory Management

In [ ]:
# For large datasets, process in chunks
def chunked_processing_example(symbol, chunk_hours=1):
    """Process large time ranges in chunks"""
    
    end = datetime.now()
    start = end - timedelta(days=1)
    
    chunk_delta = timedelta(hours=chunk_hours)
    current = start
    
    results = []
    
    while current < end:
        chunk_end = min(current + chunk_delta, end)
        
        df_chunk = repo.get_agg_trades(
            symbol=symbol,
            start_time=current,
            end_time=chunk_end,
            return_dataframe=True
        )
        
        # Process chunk
        chunk_result = {
            'start': current,
            'end': chunk_end,
            'trade_count': len(df_chunk)
        }
        results.append(chunk_result)
        
        print(f"Processed chunk: {current} to {chunk_end} ({len(df_chunk):,} trades)")
        
        current = chunk_end
    
    return results

print("💡 Best Practice: Process large datasets in chunks")
print("  - Prevents memory overflow")
print("  - Enables progress tracking")
print("  - Allows for parallel processing")

## Summary

This notebook demonstrated:

✅ **Data Access**
- Repository initialization and configuration
- Querying historical trades
- Symbol statistics

✅ **Bar Sampling**
- Dollar volume bars (fixed, adaptive, optimal)
- Time-based OHLCV candles
- Multiple intervals

✅ **Market Analysis**
- Order flow analysis
- Volume profile
- Liquidity metrics
- Microstructure patterns

✅ **Real-Time Features**
- WebSocket streaming
- Analyzer framework
- Custom analyzer development

✅ **Data Export**
- Parquet format
- CSV format
- Processed results

✅ **Best Practices**
- Error handling
- Context managers
- Memory management
- Performance optimization

### Next Steps

1. Explore the **readers notebook** for exhaustive data ingestion examples
2. Review the API reference documentation
3. Check out advanced examples in other notebooks
4. Build your own analyzers and strategies

### Resources

- [API Reference](../docs/API_REFERENCE.md)
- [RLlib Integration Specs](../specs/rllib_specs.md)
- [Notebooks README](./README.md)

---

**Happy Trading! 📈**